In [1]:
%pip install pandas sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:

import pandas as pd
import mysql.connector
from pathlib import Path
from mysql.connector import Error

DATA_DIR = Path.cwd().parent / "Datasets"
CSV_PATH = DATA_DIR / "processed" / "transactions_combined.csv"

print("CSV exists:", CSV_PATH.exists())
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="mysqlroot07",
    database="banking_analytics",
    port=3306)

cursor = conn.cursor()
df = pd.read_csv(CSV_PATH)
df.columns = (df.columns.str.strip().str.lower().str.replace(" ", "_"))
df = df.rename(columns={"account_partern_id": "account_partner_id"})
print("CSV shape:", df.shape)
print("Columns:", df.columns.tolist())
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date
df = df.astype(object).where(pd.notna(df), None)
cursor.execute("""
CREATE TABLE IF NOT EXISTS transactions_combined (
    trans_id BIGINT,
    account_id BIGINT,
    date DATE,
    type VARCHAR(30),
    operation VARCHAR(100),
    amount DECIMAL(15,2),
    balance DECIMAL(15,2),
    purpose VARCHAR(100),
    bank VARCHAR(100),
    account_partner_id BIGINT,
    source_file VARCHAR(100))""")

columns = df.columns.tolist()
column_sql = ", ".join(f"`{col}`" for col in columns)
placeholders = ", ".join(["%s"] * len(columns))
insert_sql = f"""
INSERT INTO transactions_combined ({column_sql})
VALUES ({placeholders})
"""
batch_size = 5000
try:
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start + batch_size]
        cursor.executemany(  insert_sql,list(batch.itertuples(index=False, name=None)))
        conn.commit()
        print(
            f"Inserted {min(start + batch_size, len(df))}"f" / {len(df)} rows")
    print("Transaction import completed successfully.")
except Error as e:
    conn.rollback()
    print("Import error:", e)
finally:
    cursor.close()
    conn.close()

CSV exists: True
CSV shape: (1048575, 11)
Columns: ['trans_id', 'account_id', 'date', 'type', 'operation', 'amount', 'balance', 'purpose', 'bank', 'account_partner_id', 'source_file']
Inserted 5000 / 1048575 rows
Inserted 10000 / 1048575 rows
Inserted 15000 / 1048575 rows
Inserted 20000 / 1048575 rows
Inserted 25000 / 1048575 rows
Inserted 30000 / 1048575 rows
Inserted 35000 / 1048575 rows
Inserted 40000 / 1048575 rows
Inserted 45000 / 1048575 rows
Inserted 50000 / 1048575 rows
Inserted 55000 / 1048575 rows
Inserted 60000 / 1048575 rows
Inserted 65000 / 1048575 rows
Inserted 70000 / 1048575 rows
Inserted 75000 / 1048575 rows
Inserted 80000 / 1048575 rows
Inserted 85000 / 1048575 rows
Inserted 90000 / 1048575 rows
Inserted 95000 / 1048575 rows
Inserted 100000 / 1048575 rows
Inserted 105000 / 1048575 rows
Inserted 110000 / 1048575 rows
Inserted 115000 / 1048575 rows
Inserted 120000 / 1048575 rows
Inserted 125000 / 1048575 rows
Inserted 130000 / 1048575 rows
Inserted 135000 / 1048575 rows

## creating workbook to save results

In [ ]:
import re
import sqlparse
from mysql.connector import Error
# 1. Update these paths
sql_file = Path(r"C:\Users\laptop\OneDrive\Documents\BankingFinanceAdHocAnalysis\Notebooks\sql_queries.sql")
output_file = Path(r"C:\Users\laptop\OneDrive\Documents\BankingFinanceAdHocAnalysis\Notebooks\banking_analysis_results.xlsx")

# 2. Enter your own MySQL connection details
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="mysqlroot07",
    database="banking_analytics",
    port=3306)

# 3. Read and split the SQL file into statements
sql_text = sql_file.read_text(encoding="utf-8")
statements = sqlparse.split(sql_text)

exported = 0
used_names = set()

try:
    with conn.cursor() as cursor:
        with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

            for statement in statements:

                # Capture a heading from a preceding SQL comment
                comments = re.findall(
                    r"(?m)^\s*--\s*(.+)$", statement
                )

                # Remove comments before checking the SQL command
                query = re.sub(
                    r"(?m)^\s*--.*$", "", statement
                )
                query = re.sub(
                    r"/\*.*?\*/", "", query, flags=re.S
                ).strip()

                if not query:
                    continue

                # Export read-only SELECT or WITH queries
                if not re.match(r"^(SELECT|WITH)\b", query, re.I):
                    continue

                cursor.execute(query)

                if not cursor.with_rows:
                    continue

                rows = cursor.fetchall()
                columns = [col[0] for col in cursor.description]
                df = pd.DataFrame(rows, columns=columns)

                exported += 1

                # Use a comment heading when available
                heading = comments[0] if comments else ""
                heading = re.sub(
                    r"^\s*(query\s*)?q?\d+\s*[:.)_-]*\s*",
                    "", heading, flags=re.I
                )
                heading = re.sub(
                    r"[\[\]:*?/\\]", "", heading
                ).strip()

                base_name = (
                    f"{exported:02d}_{heading}"
                    if heading else f"query_{exported:02d}"
                )
                base_name = base_name[:31].strip() or f"query_{exported:02d}"

                sheet_name = base_name
                suffix = 1

                while sheet_name.lower() in used_names:
                    ending = f"_{suffix}"
                    sheet_name = base_name[:31 - len(ending)] + ending
                    suffix += 1

                used_names.add(sheet_name.lower())

                df.to_excel(
                    writer,
                    sheet_name=sheet_name,
                    index=False
                )

                print(f"Exported {sheet_name}: {len(df)} rows")

    print(f"\nFinished! Exported {exported} query results.")
    print(f"Excel workbook: {output_file}")

finally:
    conn.close()


Exported query_01: 76 rows
Exported 02_top 5 districts by customer: 5 rows
Exported 03_average salary per district: 76 rows
Exported query_04: 2 rows
Exported query_05: 76 rows
Exported query_06: 6 rows
Exported 07_monthly transaction trends: 72 rows
Exported 08_transaction summary by type: 2 rows
Exported 09_transaction summary by opera: 6 rows
Exported query_10: 12 rows
Exported query_11: 72 rows
Exported query_12: 9 rows
Exported query_13: 20 rows
Exported query_14: 20 rows
Exported query_15: 4 rows
Exported query_16: 4 rows
Exported query_17: 3 rows
Exported query_18: 3 rows
Exported query_19: 20 rows
Exported query_20: 20 rows
Exported query_21: 76 rows
Exported query_22: 1 rows
Exported query_23: 2 rows
Exported query_24: 0 rows

Finished! Exported 24 query results.
Excel workbook: C:\Users\laptop\OneDrive\Documents\BankingFinanceAdHocAnalysis\Notebooks\banking_analysis_results.xlsx
